# Perform the statistical testing on vocabulary and style difference across considered sources

In [45]:
# Load the dataset
import pandas as pd
import numpy as np

with open("dataset/tf_idf.csv") as f:
    frequency_df = pd.read_csv(f)

stylometry_df = pd.read_csv("dataset/stylometry.csv")


In [46]:
import warnings
warnings.filterwarnings('ignore')

## On all gospels

In [47]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from auto_q.multivariate_statistical_tests import classifier_2ST
from scipy.stats import norm

In [48]:
"""Implement the Classifier Two-Sample Test.
"""
import numpy as np
from scipy.stats import norm
from sklearn.model_selection import train_test_split
import umap


def classifier_2ST(X_sample, y_sample, classifier, bootstrap=True, n_bootstraps=50):
    """Classifier Two-Sample Test.

    Parameters
    ----------
    X_sample : array-like of shape (n_samples, n_features).
        The features of the dataset.

    y_sample : array-like of shape (n_samples,).
        The labels of the dataset that one must measure.
    
    classifier : object
        A classifier object that has the methods `fit` and `predict`.

    Returns
    -------
    p_value : float
        The p-value of the test.
    """
    # Perform umap reduction on the dataset
    umap_reducer = umap.UMAP(n_components=10, random_state=30, n_neighbors=5)
    X_sample = umap_reducer.fit_transform(X_sample)
    # Compute accuracy on the original sample
    # Fit the classifier after performing train/test split
    if bootstrap:
        scores = []
        for i in range(n_bootstraps):
            X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, stratify=y_sample)
            classifier.fit(X_train, y_train)
            scores.append(classifier.score(X_test, y_test))
        score = np.mean(scores)
    else:
        X_train, X_test, y_train, y_test = train_test_split(X_sample, y_sample, stratify=y_sample)
        classifier.fit(X_train, y_train)
        score = classifier.score(X_test, y_test)
        
    print("Score of the classifier: ", score)

    r = np.mean(y_sample)
    probability_correct_classif = r**2 + (1 - r)**2
    denominator = np.sqrt(probability_correct_classif * (1 - probability_correct_classif) / len(y_sample))
    p_value = 1 - norm.cdf((score - probability_correct_classif) / denominator)

    return p_value

In [61]:
print("==== stylometry ")
# Matthew vs Luke  (stylometry)
print("=== Matthew vs Luke (stylometry) ===")
X_mt_lk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Lk")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_lk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Lk")]["book"].apply(lambda x: 1 if x == "Mt" else 0)
print(
classifier_2ST(X_mt_lk, y_mt_lk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Matthew vs Mark  (stylometry)
print("=== Matthew vs Mark (stylometry) ===")
X_mt_mk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Mk")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_mk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Mk")]["book"].apply(lambda x: 1 if x == "Mt" else 0)
print(
classifier_2ST(X_mt_mk, y_mt_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Luke vs Mark  (stylometry)
print("=== Luke vs Mark (stylometry) ===")
X_lk_mk = stylometry_df[(stylometry_df["book"] == "Lk") | (stylometry_df["book"] == "Mk")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_mk = stylometry_df[(stylometry_df["book"] == "Lk") | (stylometry_df["book"] == "Mk")]["book"].apply(lambda x: 1 if x == "Lk" else 0)
print(
classifier_2ST(X_lk_mk, y_lk_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

==== stylometry 
=== Matthew vs Luke (stylometry) ===
Score of the classifier:  0.5323076923076924
0.34029675629233547
=== Matthew vs Mark (stylometry) ===
Score of the classifier:  0.7475
0.04008188322049944
=== Luke vs Mark (stylometry) ===
Score of the classifier:  0.7199999999999999
0.11952055063215172


In [60]:
print("==== stylometry (Double) ")
# Matthew vs Luke  (stylometry)
print("=== Matthew vs Luke (stylometry) ===")
X_mt_lk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Lk") | (stylometry_df["source"] == "Double")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_lk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Lk") | (stylometry_df["source"] == "Double")]["book"].apply(lambda x: 1 if x == "Mt" else 0)
print(
classifier_2ST(X_mt_lk, y_mt_lk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Matthew vs Mark  (stylometry)
print("=== Matthew vs Mark (stylometry) ===")
X_mt_mk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_mk = stylometry_df[(stylometry_df["book"] == "Mt") | (stylometry_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")]["book"].apply(lambda x: 1 if x == "Mt" else 0)
print(
classifier_2ST(X_mt_mk, y_mt_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Luke vs Mark  (stylometry)
print("=== Luke vs Mark (stylometry) ===")
X_lk_mk = stylometry_df[(stylometry_df["book"] == "Lk") | (stylometry_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_mk = stylometry_df[(stylometry_df["book"] == "Lk") | (stylometry_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")]["book"].apply(lambda x: 1 if x == "Lk" else 0)
print(
classifier_2ST(X_lk_mk, y_lk_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

==== stylometry (Double) 
=== Matthew vs Luke (stylometry) ===
Score of the classifier:  0.5030769230769232
0.5006944010505983
=== Matthew vs Mark (stylometry) ===
Score of the classifier:  0.5920000000000001
0.15346753266256163
=== Luke vs Mark (stylometry) ===
Score of the classifier:  0.6454545454545455
0.05601129544383676


In [65]:
print("==== vocabulary ")
# Matthew vs Luke  (vocabulary)
print("=== Matthew vs Luke (vocabulary) ===")
X_mt_lk = frequency_df[(frequency_df["book"] == "Mt") | (frequency_df["book"] == "Lk") | (stylometry_df["source"] == "Double")].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_lk = frequency_df[(frequency_df["book"] == "Mt") | (frequency_df["book"] == "Lk") | (stylometry_df["source"] == "Double")]["book"].apply(lambda x: 0 if x == "Mt" else 1)
print(
classifier_2ST(X_mt_lk, y_mt_lk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Matthew vs Mark  (vocabulary)
print("=== Matthew vs Mark (vocabulary)===")
X_mt_mk = frequency_df[(frequency_df["book"] == "Mt") | (frequency_df["book"] == "Mk") | (stylometry_df["source"] == "Triple") ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_mk = frequency_df[(frequency_df["book"] == "Mt") | (frequency_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")]["book"].apply(lambda x: 0 if x == "Mt" else 1)
print(
classifier_2ST(X_mt_mk, y_mt_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

# Luke vs Mark  (vocabulary)
print("=== Luke vs Mark (vocabulary)===")
X_lk_mk = frequency_df[(frequency_df["book"] == "Lk") | (frequency_df["book"] == "Mk") | (stylometry_df["source"] == "Triple") ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_mk = frequency_df[(frequency_df["book"] == "Lk") | (frequency_df["book"] == "Mk") | (stylometry_df["source"] == "Triple")]["book"].apply(lambda x: 0 if x == "Lk" else 1)
print(
classifier_2ST(X_lk_mk, y_lk_mk, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)


==== vocabulary 
=== Matthew vs Luke (vocabulary) ===
Score of the classifier:  0.46307692307692305
0.7147915505414348
=== Matthew vs Mark (vocabulary)===
Score of the classifier:  0.54
0.358020851066419
=== Luke vs Mark (vocabulary)===
Score of the classifier:  0.5727272727272728
0.2666286340169355


# On Matthew's gospel

In [57]:
# In Matthew
print("=== In Matthew difference between the Double and the Triple tradition (style)")
X_mt_double_triple = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "Double")| (stylometry_df["source"] == "Triple")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_double_triple = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "Double")| (stylometry_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "Double" else 0)
print(
classifier_2ST(X_mt_double_triple, y_mt_double_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Matthew difference between the M and the triple tradition (style)")
X_mt_M_triple = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "M")| (stylometry_df["source"] == "Triple")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_M_triple = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "M")| (stylometry_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "M" else 0)
print(
classifier_2ST(X_mt_M_triple, y_mt_M_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Matthew difference between the M and the double tradition (style)")
X_mt_M_double = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "M")| (stylometry_df["source"] == "Double")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_M_double = stylometry_df[(stylometry_df["book"] == "Mt") & ((stylometry_df["source"] == "M")| (stylometry_df["source"] == "Double"))]["source"].apply(lambda x: 1 if x == "M" else 0)
print(
    classifier_2ST(X_mt_M_double, y_mt_M_double, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)


print("=== In Matthew difference between the double and the triple tradition (vocabulary)")
X_mt_double_triple = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Double")| (frequency_df["source"] == "Triple")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_double_triple = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Double")| (frequency_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "Double" else 0)
print(
classifier_2ST(X_mt_double_triple, y_mt_double_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Matthew difference between the M and the triple tradition (vocabulary)")
X_mt_M_triple = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Triple")| (frequency_df["source"] == "M")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_M_triple = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Triple")| (frequency_df["source"] == "M"))]["source"].apply(lambda x: 1 if x == "Triple" else 0)
print(
classifier_2ST(X_mt_M_triple, y_mt_M_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Matthew difference between the M and the double tradition (vocabulary)")
X_mt_M_double = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Double")| (frequency_df["source"] == "M")) ].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_mt_M_double = frequency_df[(frequency_df["book"] == "Mt") & ((frequency_df["source"] == "Double")| (frequency_df["source"] == "M"))]["source"].apply(lambda x: 1 if x == "Double" else 0)
print(
classifier_2ST(X_mt_M_double, y_mt_M_double, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)


=== In Matthew difference between the Double and the Triple tradition (style)
Score of the classifier:  0.54
0.46153412738544375
=== In Matthew difference between the M and the triple tradition (style)
Score of the classifier:  0.596
0.22295038488637842
=== In Matthew difference between the M and the double tradition (style)
Score of the classifier:  0.45
0.7946675559776016
=== In Matthew difference between the double and the triple tradition (vocabulary)
Score of the classifier:  0.5
0.5762403058717522
=== In Matthew difference between the M and the triple tradition (vocabulary)
Score of the classifier:  0.688
0.06141875945259612
=== In Matthew difference between the M and the double tradition (vocabulary)
Score of the classifier:  0.595
0.37925461646833236


## On Luke's Gospel

In [58]:
# In Luke
print("=== In Luke difference between the Double and the Triple tradition (style)")
X_lk_double_triple = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "Double") | (stylometry_df["source"] == "Triple"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_double_triple = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "Double") | (stylometry_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "Double" else 0)
print(
    classifier_2ST(X_lk_double_triple, y_lk_double_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Luke difference between the L and the Triple tradition (style)")
X_lk_L_triple = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "L") | (stylometry_df["source"] == "Triple"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_L_triple = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "L") | (stylometry_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "L" else 0)
print(
    classifier_2ST(X_lk_L_triple, y_lk_L_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Luke difference between the L and the Double tradition (style)")
X_lk_L_double = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "L") | (stylometry_df["source"] == "Double"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_L_double = stylometry_df[(stylometry_df["book"] == "Lk") & ((stylometry_df["source"] == "L") | (stylometry_df["source"] == "Double"))]["source"].apply(lambda x: 1 if x == "L" else 0)
print(
    classifier_2ST(X_lk_L_double, y_lk_L_double, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Luke difference between the Double and the Triple tradition (vocabulary)")
X_lk_double_triple = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "Double") | (frequency_df["source"] == "Triple"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_double_triple = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "Double") | (frequency_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "Double" else 0)
print(
    classifier_2ST(X_lk_double_triple, y_lk_double_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Luke difference between the L and the Triple tradition (vocabulary)")
X_lk_L_triple = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "L") | (frequency_df["source"] == "Triple"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_L_triple = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "L") | (frequency_df["source"] == "Triple"))]["source"].apply(lambda x: 1 if x == "L" else 0)
print(
    classifier_2ST(X_lk_L_triple, y_lk_L_triple, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)

print("=== In Luke difference between the L and the Double tradition (vocabulary)")
X_lk_L_double = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "L") | (frequency_df["source"] == "Double"))].drop(["parable", "source", "source_interpretation", "book"], axis=1)
y_lk_L_double = frequency_df[(frequency_df["book"] == "Lk") & ((frequency_df["source"] == "L") | (frequency_df["source"] == "Double"))]["source"].apply(lambda x: 1 if x == "L" else 0)
print(
    classifier_2ST(X_lk_L_double, y_lk_L_double, classifier=RandomForestClassifier(n_estimators=50, random_state=12))
)


=== In Luke difference between the Double and the Triple tradition (style)
Score of the classifier:  0.435
0.7459093379399658
=== In Luke difference between the L and the Triple tradition (style)
Score of the classifier:  0.7
0.06281863394305798
=== In Luke difference between the L and the Double tradition (style)
Score of the classifier:  0.5680000000000001
0.6536909201950586
=== In Luke difference between the Double and the Triple tradition (vocabulary)
Score of the classifier:  0.53
0.49028430966788905
=== In Luke difference between the L and the Triple tradition (vocabulary)
Score of the classifier:  0.6033333333333334
0.2669044543632644
=== In Luke difference between the L and the Double tradition (vocabulary)
Score of the classifier:  0.7519999999999999
0.10551461229927517
